# layax on Kaggle (2x T4)

Late-interaction option heads + a Learn-then-Test abstention gate for [Laya](https://github.com/NandhaKishorM/laya).

**What this notebook does**
1. Installs dependencies and the `layax` package.
2. Downloads a Laya checkpoint and initialises the late-interaction model from it.
3. Trains on Banking77 (77 labels — the case upstream loses on).
4. Fits temperatures, then the abstention gate on the calibration split.
5. Evaluates against upstream Laya arms and writes a report.

**On the two T4s.** The trainer is single-GPU by design. `DataParallel` at this batch size
spends more time copying than it saves, so the second T4 is used for the baseline arms
instead, which run concurrently on `cuda:1`. If you want true multi-GPU training, use
`torchrun` with DDP rather than DP.

**T4 means fp16.** T4 is Turing, so there is no bf16. `amp_dtype` stays `fp16` and the
trainer keeps a `GradScaler`. On an Ampere or newer card, switch to `bf16` and the
scaler turns itself off.

**Budget.** Banking77 at 3 epochs is roughly 40–70 minutes on one T4. Kaggle's session
limit is 12 hours, so the sweep at the bottom fits; a 4-epoch run over ~30k questions,
as upstream's own fine-tuning notebook does, takes 4–5 hours.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), "capability", torch.cuda.get_device_capability(i))

## 1. Install

`laya` itself is only needed for the baseline arms. Everything in `layax` runs without it.

In [ ]:
!pip install -q "transformers>=4.48" "safetensors>=0.4" "huggingface_hub>=0.25" \
    "datasets>=2.19" "scipy>=1.11" laya 2>&1 | tail -3

## 2. Get the `layax` package onto the machine

Three ways, in order of preference. Upload the starter as a Kaggle Dataset named `layax`
and the first branch just works.

In [ ]:
import os, sys, shutil, subprocess

DEST = "/kaggle/working/layax"
if os.path.isdir("/kaggle/input/layax/layax"):
    shutil.copytree("/kaggle/input/layax/layax", DEST, dirs_exist_ok=True)
    shutil.copytree("/kaggle/input/layax/configs", "/kaggle/working/configs", dirs_exist_ok=True)
    print("using uploaded dataset")
elif os.environ.get("LAYAX_GIT"):
    subprocess.run(["git", "clone", "-q", os.environ["LAYAX_GIT"], "/kaggle/working/src"], check=True)
    shutil.copytree("/kaggle/working/src/layax", DEST, dirs_exist_ok=True)
    shutil.copytree("/kaggle/working/src/configs", "/kaggle/working/configs", dirs_exist_ok=True)
    print("using git clone")
else:
    raise SystemExit("Upload the starter as a Kaggle Dataset called 'layax', "
                     "or set LAYAX_GIT to a clone URL.")

sys.path.insert(0, "/kaggle/working")
import layax
print("layax", layax.__version__)

## 3. Shape check before spending GPU time

Runs on random weights in about a second. If this fails, nothing downstream is worth
debugging — it separates a broken environment from a broken model.

In [ ]:
!cd /kaggle/working && python -m layax.cli smoke

## 4. Hugging Face token (optional)

Only needed if the checkpoint download is rate-limited or gated. Add it as a Kaggle
Secret called `HF_TOKEN`.

In [ ]:
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("token loaded")
except Exception as e:
    print("no token; continuing anonymously (%s)" % type(e).__name__)

## 5. Configure the run

`train_options_per_row` is the setting that makes 77 labels fit on a T4: training scores
the gold option plus 15 sampled negatives, while **evaluation always scores all 77**. Set
it to 0 to train on the full label set — correct, and far slower.

Start with `max_train_rows` small to confirm the loop runs end to end, then remove it.

In [ ]:
from layax.config import RunConfig, LIConfig, CompConfig

cfg = RunConfig(
    name="banking77-t4",
    dataset="banking77",
    output_dir="/kaggle/working/runs/banking77",
    li=LIConfig(
        base_checkpoint="convaiinnovations/laya",
        base_subfolder="typed-decisions",   # the only checkpoint above chance on typed decisions
        interaction="maxsim",               # "xattn" is the more expressive arm
        proj_dim=128,
        state_max_len=512,
        option_max_len=24,
        epochs=3,
        batch_size=8,
        grad_accum=4,                       # effective batch 32
        amp_dtype="fp16",                   # T4: no bf16
        train_options_per_row=16,
        in_batch_negatives=True,
    ),
    comp=CompConfig(
        shift_foreign=0.25,                 # languages the checkpoint cannot read
        shift_truncate=0.15,
        shift_distractors=0.20,
        target_risk=0.05,                   # gate must hold selective risk <= 5%
        min_coverage=0.30,
    ),
    competence_frac=0.15,
    calibration_frac=0.10,
    max_train_rows=None,                    # set e.g. 2000 for a fast dry run
    max_eval_rows=3000,
)
cfg.validate()
print(cfg.to_dict()["li"]["interaction"], "|", cfg.splits if hasattr(cfg, "splits") else "")

## 6. Train and evaluate

Everything runs through `layax.pipeline.run`, the same entrypoint Modal uses, so what you
measure here is what runs there.

In [ ]:
from layax.pipeline import run

report = run(cfg, device="cuda:0", hf_token=HF_TOKEN, run_baselines=False)
print(report["table"])

## 7. Baselines on the second T4

Upstream arms, so the comparison is against Laya's best available answer for wide
label sets rather than only the configuration it is known to fail on:

- plain `laya` at default settings
- `laya` + `shortlist(k)` for k in 10, 20, 25, 40 — the retrieval workaround shipped in
  `laya.shortlist`, with recall@k (is the gold in the shortlist) next to accuracy
- `laya` with `head_max_len=512` — the runtime widening the README suggests for 50+ options

In [ ]:
from layax.evaluate import LAYA_BASELINE_ARMS, evaluate_laya_baseline, compare
from layax.pipeline import load_splits

splits = load_splits(cfg)
rows = splits.test[:1500]

arms = []
for kw in LAYA_BASELINE_ARMS:
    try:
        arms.append(evaluate_laya_baseline(
            rows, cfg.li.base_checkpoint, cfg.li.base_subfolder, device="cuda:1", **kw))
    except Exception as e:
        arms.append({"arm": "laya %s" % (kw or "default"), "n": 0, "errors": str(e)[:100]})

m = report["test"]["metrics"]
arms.append({"arm": "layax-maxsim", "n": m["n"], "accuracy": m["accuracy"], "coverage": 1.0,
             "ece": m["ece_softmax"], "aurc": m["aurc_softmax"], "errors": 0})
print(compare(arms))

## 8. The abstention gate

The default gate is Learn-then-Test on the temperature-scaled max-softmax. The threshold is
fitted on the calibration split and bounds P(selective risk <= target) >= 1 - delta, only
on data like the calibration split. If no threshold meets the target, `feasible` is false
and the model never abstains on that basis. The learned competence head is off by default
(`competence_head: false`); it lost to max-softmax in every run.

In [ ]:
import json
print("fit on calibration:", json.dumps(report["msp_gate"], indent=2))
print("on test:", json.dumps(report["test"]["gate"], indent=2))

## 9. Behaviour under shift

Shifted test rows (truncated inputs, foreign scripts) are not covered by the bound. These
numbers are empirical only.

In [ ]:
m = report["test"]["metrics"]
print("full accuracy %.3f | AURC %.4f | ECE %.4f" % (m["accuracy"], m["aurc_softmax"], m["ece_softmax"]))
print(json.dumps(report["test_shift"]["by_shift"], indent=2))

## 10. Latency: cold vs warm cache

Warm is the production path — options encoded once per schema, so each request is one
state pass plus a matmul. Cold is what the first request after a deploy sees. Report
both: upstream measures 7–10 s on a language switch at default settings, and the
difference between those two numbers is the entire argument for preloading.

In [ ]:
from layax.evaluate import latency_benchmark
from layax.runtime import LayaxAgent

agent = LayaxAgent.from_pretrained(cfg.output_dir, device="cuda:0")
for n in (3, 20, 77, 150):
    qs = {"intent": {"type": "choice", "instructions": "Which intent is this?",
                     "criteria": {"label_%03d" % i: "intent number %d" % i for i in range(n)}}}
    r = latency_benchmark(agent, qs, "i lost my card and need a replacement", repeats=30)
    print("%4d options | cold %8.1f ms | warm p50 %6.2f ms | p95 %6.2f ms"
          % (n, r["cold_ms"], r["p50_ms"], r["p95_ms"]))

## 11. Save artifacts

Everything under `/kaggle/working` is downloadable when the session ends.

In [ ]:
import shutil, os
print(os.listdir(cfg.output_dir))
shutil.make_archive("/kaggle/working/layax-run", "zip", cfg.output_dir)
print("wrote /kaggle/working/layax-run.zip")

## 12. What to run next

- **Swap the dataset for your own rows.** Set `dataset="jsonl"` and point `--jsonl` at a
  file of `{"state": ..., "label": ..., "criteria": {...}}` lines. A support-request
  taxonomy is the same high-cardinality shape as Banking77.
- **Two-stage taxonomies.** Category then subcategory beats one flat 200-option question,
  and the head makes the second stage cheap because option embeddings are cached.
- **Ablate.** `scripts/modal_app.py::sweep` runs maxsim vs xattn, negatives on/off, hard
  vs random sampling, and shift on/off in parallel. The `no-shift` arm is the one that
  tells you whether shift augmentation is doing anything.
- **Check novelty before claiming any.** Late interaction and correctness predictors are
  established techniques. The new part is applying them to Laya's RLCD-trained heads —
  search recent work before writing that up as new.